In [2]:
!pip install -q transformers datasets evaluate accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [3]:
## Preparing Data and Tokenization
## We'll load a sample dataset (or create a small custom classification dataset) and prepare tokenized inputs.

from datasets import Dataset
import pandas as pd
from transformers import AutoTokenizer

# 1. Create a sample supervised dataset (e.g., customer support routing)
data = {
    "text": [
        "My payment failed and I was charged twice.",
        "How do I update my profile picture?",
        "The app keeps crashing whenever I open settings.",
        "I need a receipt for my subscription purchase.",
        "Can I change my email address on file?",
        "The screen turns black on startup."
      ],
      "label": [0, 1, 2, 0, 1, 2] # 0: Billing, 1: Account, 2: Technical
}

dataset = Dataset.from_dict(data)

# 2. Load Tokenizer
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

# 3. Tokenization Function
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=32)

tokenized_dataset = dataset.map(tokenize, batched=True)
print("Tokenized Dataset Sample:\n", tokenized_dataset[0])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenized Dataset Sample:
 {'text': 'My payment failed and I was charged twice.', 'label': 0, 'input_ids': [101, 2026, 7909, 3478, 1998, 1045, 2001, 5338, 3807, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [4]:
## Defining Model Architecture & Evaluation Metrics
## Attach a 3-class classification head to the pretrained distilbert-base-uncased backbone.
import numpy as np
import evaluate
from transformers import AutoModelForSequenceClassification

# 1. Load Pretrained Model with Classification Head
num_labels = 3
id2label = {0: "Billing", 1: "Account", 2: "Technical"}
label2id = {"Billing": 0, "Account": 1, "Technical": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id

)

# 2. Define Metric Calculation
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
## Training with Hugging Face Trainer
## Set up hyperparameters and train the model end-to-end.
from transformers import TrainingArguments, Trainer

# 1. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_steps=1,
    learning_rate=2e-5,
    save_strategy="no",
    report_to="none"
)

# 2. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,
    compute_metrics=compute_metrics
 )

# 3. Start Fine-Tuning
print("--- STARTING SUPERVISED FINE-TUNING ---")
trainer.train()

--- STARTING SUPERVISED FINE-TUNING ---


Step,Training Loss
1,1.174277
2,1.133980
3,1.107169
4,1.105549
5,1.137752
6,1.041344
7,1.111941
8,1.082065
9,1.049865


TrainOutput(global_step=9, training_loss=1.104882346259223, metrics={'train_runtime': 1.2375, 'train_samples_per_second': 14.545, 'train_steps_per_second': 7.273, 'total_flos': 149028481152.0, 'train_loss': 1.104882346259223, 'epoch': 3.0})

In [7]:
## Inference with Fine-Tuned Model
## Test the fine-tuned classifier on unseen text inputs.
from transformers import pipeline

# Wrap fine-tuned model into a classification pipeline
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

new_text = "I cannot log into my account after resetting the password."
prediction = classifier(new_text)

print("--- CLASSIFICATION INFERENCE ---")
print(f"Text: '{new_text}'")
print(f"Predicted Class: {prediction[0]['label']} (Score: {prediction[0]['score']:.4f})")

--- CLASSIFICATION INFERENCE ---
Text: 'I cannot log into my account after resetting the password.'
Predicted Class: Billing (Score: 0.3454)
